# 04 Array source: a 256-element focused array and a small bone sphere

| | |
|---|---|
| **Purpose** | A realistic therapeutic set-up: a multi-element spherical array at 500 kHz focused on a 7.5 mm bone sphere. Shows how array element positions are loaded from a file and how to inspect the incident field before paying for a BEM solve. |
| **Inputs** | 500 kHz; 256 circular elements of radius 3 mm on a spherical cap of radius of curvature 0.18 m (centroids from `notebooks/Data/default_random_array_centroid_locations.dat`), focus at the origin, normal velocity 0.4 m/s; bone sphere of radius 7.5 mm at the focus in water, element size 0.75 mm (full) or 1.0 mm (CI). |
| **Outputs** | Array layout; incident pressure magnitude near the focus; total pressure around the sphere (`outputs/04_array_source_*.png`). |
| **Checks** | Incident focal peak within one wavelength of the geometric focus; lateral -6 dB width within 20% of the focused-aperture estimate $1.41 \lambda F_{\#}$; GMRES converged; field finite. |
| **Runtime** | About 3 min full (77 s of it the solve, 1607 vertices), about 70 s CI mode with the 1.0 mm mesh (Docker image, 4 CPUs, 9 Nov 2026). |
| **Adapted from** | `notebooks/Array source simulation.ipynb` (upstream OptimUS). |

## Physics notes

Each element is a small baffled piston (sampled with point sources, as in `02_piston_source`), and the
array field is the sum of the element fields. Because the elements sit on a sphere of radius
$d = 0.18$ m centred on the origin, all their waves arrive in phase there: the geometric focus.

For a focused circular aperture of diameter $D$ the pressure across the focal plane follows the
Airy pattern $2 J_1(x)/x$ with $x = \pi D r / (\lambda d)$, so the pressure drops to half
(-6 dB) at a full width of about $1.41 \, \lambda \, F_{\#}$, where $F_{\#} = d / D$
(Cobbold, *Foundations of Biomedical Ultrasound*, 2007, chapter 4). A sparse random array is only
approximately a continuous aperture, so the check allows 20%.

> **To expand (Yash):** derive the 1.41 factor from $2J_1(x)/x = 0.5$; discuss grating lobes and why the element layout is random.

## Setup

Imports, the `CI_MODE` switch and the output folder.

In [ ]:
NOTEBOOK_NAME = "04_array_source"

%matplotlib inline
import json
import os
import time

import numpy as np
import matplotlib.pyplot as plt

import optimus

# CI_MODE uses smaller settings so the notebook finishes in a few minutes in CI.
# Set it from the shell with OPTIMUS_CI_MODE=1, or change it here by hand.
CI_MODE = os.environ.get("OPTIMUS_CI_MODE", "0") == "1"

# Figures and a small JSON file of key results are written here.
OUTPUT_DIR = os.environ.get("OPTIMUS_OUTPUT_DIR", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS = {"notebook": NOTEBOOK_NAME, "ci_mode": CI_MODE,
           "optimus_version": optimus.__version__, "checks": {}}
START_TIME = time.time()

print("OptimUS", optimus.__version__, "| CI mode:", CI_MODE, "| outputs in:", OUTPUT_DIR)

In [ ]:
def check(name, condition, detail):
    """Record a named check and stop the notebook if it fails."""
    RESULTS["checks"][name] = bool(condition)
    print(("PASS  " if condition else "FAIL  ") + name + ": " + detail)
    assert condition, name + " failed: " + detail


def gmres_converged(model):
    """GMRES stops early (before maxiter) only when the tolerance is reached."""
    if hasattr(model, "linear_solve_iteration_count"):
        iterations = model.linear_solve_iteration_count
    else:
        iterations = model.iteration_count
    return iterations, iterations < optimus.global_parameters.linalg.maxiter


def save_figures(figs, stem):
    """Save a list of matplotlib figures as PNG files in OUTPUT_DIR."""
    if not isinstance(figs, (list, tuple)):
        figs = [figs]
    for i, fig in enumerate(figs):
        fig.savefig(os.path.join(OUTPUT_DIR, "%s_%d.png" % (stem, i + 1)), dpi=110,
                    bbox_inches="tight")

## 1. Parameters

In [ ]:
frequency = 500e3                      # Hz
element_radius = 3e-3                  # m
radius_of_curvature = 0.18             # m, array apex placed at (-d, 0, 0)
centroid_file = os.path.join("..", "Data", "default_random_array_centroid_locations.dat")
velocity = 0.4                         # m/s
sphere_radius = 7.5e-3                 # m
element_size = 1.0e-3 if CI_MODE else 0.75e-3
focal_grid = (61, 61) if CI_MODE else (121, 121)
field_grid = (100, 50) if CI_MODE else (200, 100)

## 2. Source, geometry and materials

In [ ]:
source = optimus.source.create_array(frequency, element_radius=element_radius,
                                     centroid_locations_filename=centroid_file,
                                     location=[-radius_of_curvature, 0, 0],
                                     number_of_point_sources_per_wavelength=2, velocity=velocity)
geometry = optimus.geometry.shapes.Sphere(radius=sphere_radius, element_size=element_size)
material_ext = optimus.material.load_material("water")
material_int = optimus.material.load_material("bone (cortical)")

wavelength = material_ext.compute_wavelength(frequency)
centroids = source.centroid_locations
aperture_diameter = np.ptp(centroids[1]) + 2 * element_radius
f_number = radius_of_curvature / aperture_diameter
print("%d elements, aperture about %.1f mm, F-number %.2f, wavelength %.2f mm"
      % (centroids.shape[1], aperture_diameter * 1e3, f_number, wavelength * 1e3))
print("Sphere mesh: %d vertices, %.1f elements per wavelength"
      % (geometry.number_of_vertices(), wavelength / element_size))
RESULTS["number_of_vertices"] = int(geometry.number_of_vertices())

fig = plt.figure(figsize=(5, 5))
plt.scatter(centroids[1] * 1e3, centroids[2] * 1e3, s=8)
plt.gca().set_aspect("equal")
plt.xlabel("y [mm]")
plt.ylabel("z [mm]")
plt.title("Element centroids seen from the focus")
save_figures(fig, NOTEBOOK_NAME + "_layout")
plt.show()

## 3. Incident field near the focus

Cheap compared with the BEM solve, and worth checking first: if the focus is in the wrong place the
solve is wasted.

In [ ]:
half_width = 10e-3
xs = np.linspace(-half_width, half_width, focal_grid[0])
ys = np.linspace(-half_width, half_width, focal_grid[1])
X, Y = np.meshgrid(xs, ys, indexing="ij")
points = np.vstack((X.ravel(), Y.ravel(), 0 * X.ravel()))
p_inc = np.abs(source.pressure_field(material_ext, points)).reshape(X.shape)

i_peak = np.unravel_index(np.argmax(p_inc), p_inc.shape)
peak_offset = np.hypot(xs[i_peak[0]], ys[i_peak[1]])

y_line = np.linspace(-6e-3, 6e-3, 241)
p_line = np.abs(source.pressure_field(material_ext, np.vstack((0 * y_line + xs[i_peak[0]], y_line, 0 * y_line))))
above_half = y_line[p_line >= 0.5 * p_line.max()]
width_6db = above_half.max() - above_half.min()
width_expected = 1.41 * wavelength * f_number
RESULTS["incident_peak_MPa"] = round(float(p_inc.max() / 1e6), 3)
RESULTS["focal_peak_offset_mm"] = round(float(peak_offset * 1e3), 2)
RESULTS["lateral_6dB_width_mm"] = round(float(width_6db * 1e3), 2)
RESULTS["lateral_6dB_width_expected_mm"] = round(float(width_expected * 1e3), 2)
print("Incident peak %.2f MPa, %.2f mm from the geometric focus" % (p_inc.max() / 1e6, peak_offset * 1e3))
print("Lateral -6 dB width %.2f mm, estimate 1.41 lambda F# = %.2f mm" % (width_6db * 1e3, width_expected * 1e3))

fig = plt.figure(figsize=(6, 5))
plt.imshow(p_inc.T / 1e6, extent=[-half_width * 1e3, half_width * 1e3, -half_width * 1e3, half_width * 1e3],
           origin="lower", cmap="viridis")
plt.colorbar(label="|p_inc| [MPa]")
theta = np.linspace(0, 2 * np.pi, 100)
plt.plot(sphere_radius * 1e3 * np.cos(theta), sphere_radius * 1e3 * np.sin(theta), "w--", lw=1)
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.title("Incident field at the focus (dashed: bone sphere)")
save_figures(fig, NOTEBOOK_NAME + "_incident")
plt.show()

## 4. Model and solve

In [ ]:
model = optimus.model.create_default_model(source, geometry, material_ext, material_int)
t0 = time.time()
model.solve()
RESULTS["solve_time_s"] = round(time.time() - t0, 1)
iterations, converged = gmres_converged(model)
RESULTS["gmres_iterations"] = int(iterations)
print("GMRES iterations:", iterations, "| solve time %.1f s" % RESULTS["solve_time_s"])

## 5. Total field around the sphere

In [ ]:
plane = optimus.postprocess.VisualisePlane(model)
plane.create_computational_grid(resolution=field_grid, bounding_box=(-0.06, 0.06, -0.03, 0.03))
plane.compute_fields()
figs = optimus.postprocess.plot_pressure_field(plane, field="total", clim=(-10, 10), unit="MPa")
save_figures(figs, NOTEBOOK_NAME + "_total")

## 6. Checks

Measured on 9 Nov 2026: incident peak 1.7 mm before the geometric focus (wavelength 2.96 mm) and a
lateral -6 dB width of 4.5 mm against the estimate of 4.8 mm (full mode).

In [ ]:
check("focus_in_place", peak_offset < wavelength,
      "peak %.2f mm from the focus, limit one wavelength (%.2f mm)" % (peak_offset * 1e3, wavelength * 1e3))
check("focal_width", abs(width_6db / width_expected - 1) < 0.2,
      "width %.2f mm vs %.2f mm expected (limit 20%%)" % (width_6db * 1e3, width_expected * 1e3))
check("gmres_converged", converged, "%d iterations" % iterations)
total = plane.total_field_imshow
check("field_is_finite", np.all(np.isfinite(total)),
      "%d NaN or inf values on the plane" % np.sum(~np.isfinite(total)))

## Summary

Key numbers and check results, also saved as JSON in `OUTPUT_DIR` so CI and the assistant can read them.

In [ ]:
RESULTS["runtime_s"] = round(time.time() - START_TIME, 1)
with open(os.path.join(OUTPUT_DIR, NOTEBOOK_NAME + "_results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2, sort_keys=True)
print(json.dumps(RESULTS, indent=2, sort_keys=True))